# Supertonic 3 — Voice Clone Studio (Google Colab GPU Workspace)

Web GUI for training and speech generation using the Supertonic3 model on a Google Colab T4 GPU.

### Steps to Run:
1. Connect the runtime to a **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).
2. Run the cells below sequentially.
3. Open the **Ngrok** tunnel link generated at the end to access the Web UI.

In [ ]:
# ==========================================
# 1. CLONE REPOSITORY FROM GITHUB
# ==========================================
import os
import shutil
from google.colab import drive

%cd /content
print(">> Cloning repository from GitHub...")
!git clone https://github.com/dev-rexpro/supertonic-studio.git

print(">> Connecting to Google Drive to retrieve the model folder...")
drive.mount('/content/drive')

drive_path = "/content/drive/MyDrive"
target_model_dir = "/content/supertonic-studio/supertonic3"

# Check if model exists as folder or zip in Google Drive
if os.path.exists(os.path.join(drive_path, "supertonic3")):
    print(">> Copying supertonic3 model folder from Google Drive...")
    shutil.copytree(os.path.join(drive_path, "supertonic3"), target_model_dir)
    print(">> Model folder successfully copied!")
elif os.path.exists(os.path.join(drive_path, "supertonic3.zip")):
    print(">> Extracting supertonic3 model folder from supertonic3.zip in Google Drive...")
    !mkdir -p /content/temp_extract
    !unzip -q /content/drive/MyDrive/supertonic3.zip -d /content/temp_extract
    
    # Move the model folder into the cloned project
    found = False
    for root, dirs, files in os.walk("/content/temp_extract"):
        if "onnx" in dirs and os.path.basename(root) == "supertonic3":
            shutil.move(root, target_model_dir)
            found = True
            break
            
    !rm -rf /content/temp_extract
    if found:
        print(">> Model folder successfully extracted!")
    else:
        print(">> WARNING: Could not find 'supertonic3' folder inside the zip!")
else:
    print(">> WARNING: 'supertonic3' model folder or 'supertonic3.zip' not found in Drive root!")

%cd /content/supertonic-studio
print(f">> Current working directory: {os.getcwd()}")

In [ ]:
# ==========================================
# 2. INSTALL DEPENDENCIES & SETUP GPU T4
# ==========================================
print(">> Installing base dependencies...")
!pip install -r requirements.txt

print(">> Force-installing PyTorch CUDA version for T4 GPU...")
!pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

print(">> Uninstalling conflicting packages (torchcodec)...")
!pip uninstall torchcodec -y

print(">> Installing FastAPI, Uvicorn, Python-Multipart, and Pyngrok...")
!pip install fastapi uvicorn python-multipart pyngrok

In [ ]:
# ==========================================
# 3. PATCH AUTOMATIC BYPASS (SOUNDFILE)
# ==========================================
print(">> Patching utils/loss.py to use soundfile backend...")
loss_path = "utils/loss.py"

if os.path.exists(loss_path):
    with open(loss_path, "r") as f:
        code = f.read()

    new_func = """def load_audio_16khz_mono(file_path):
    import soundfile as sf
    data, sample_rate = sf.read(file_path)
    signal = torch.FloatTensor(data)
    if len(signal.shape) > 1:
        signal = torch.mean(signal, dim=1, keepdim=True)
    else:
        signal = signal.unsqueeze(0)
    if signal.shape[0] > signal.shape[1] and signal.shape[1] == 1:
        signal = signal.T
    if sample_rate != 16000:
        signal = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(signal)
    return signal"""

    lines = code.split('\n')
    new_lines = []
    skip = False
    for line in lines:
        if "def load_audio_16khz_mono" in line:
            skip = True
            new_lines.append(new_func)
            continue
        if skip and (line.startswith("class ") or line.startswith("import ") or line.startswith("from ")):
            skip = False
        if not skip:
            new_lines.append(line)

    with open(loss_path, "w") as f:
        f.write('\n'.join(new_lines))
    print(">> utils/loss.py SUCCESSFULLY PATCHED!")
else:
    print(">> ERROR: utils/loss.py not found!")

In [ ]:
# ==========================================
# 4. START NGROK TUNNEL & FASTAPI SERVER
# ==========================================
from pyngrok import ngrok

print(">> Setting Ngrok auth token...")
ngrok.set_auth_token("3FUfeyBIqJutJdK5iZ9F9mlz9sI_7QjY5AcWcV6jkrKbXSEwE")

print(">> Starting Ngrok tunnel on port 8000...")
try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(8000)
print("\n" + "="*80)
print(f"🎉 WEB INTERFACE ACTIVE!")
print(f"👉 OPEN WEB UI: {public_url}")
print("="*80 + "\n")

print(">> Running FastAPI Server...")
!python server.py